# Assembly Line Balancing — RPW e LCR

Implementazione delle due euristiche di bilanciamento della linea di assemblaggio:

- **RPW** — *Ranked Positional Weight*
- **LCR** — *Largest Candidate Rule*

**Dati del caso studio**
- 40 task del processo di assemblaggio (file `tabella_attivita_tempo_standard_MOST.xlsx`)
- Cycle Time **CT = 420 s** per stazione
- Work Content **WC = 1905 s** (somma dei tempi standard)

**Obiettivo:** partendo dalla configurazione attuale della linea (**AS-IS**, il benchmark), assegnare i task alle stazioni con RPW e LCR rispettando i vincoli di precedenza e il Cycle Time, quindi confrontare le tre configurazioni tramite i KPI.

> Le funzioni di calcolo sono raccolte nel modulo `alb_helpers.py` e vengono richiamate dalle celle di questo notebook.

In [1]:
# autoreload: ricarica automaticamente alb_helpers.py ad ogni modifica,
# così non serve riavviare il kernel quando aggiorni le funzioni helper
%load_ext autoreload
%autoreload 2

import pandas as pd
import alb_helpers as H

pd.set_option("display.max_colwidth", None)  # mostra per intero la colonna dei task

XLSX = "tabella_attivita_tempo_standard_MOST.xlsx"
CT = H.CT_DEFAULT      # Cycle Time = 420 s
print(f"Cycle Time CT = {CT} s")

Cycle Time CT = 420 s


## 1. Struttura dei dati

Carichiamo la tabella dei task dal foglio Excel. La colonna *Predecessori* viene trasformata da stringa a lista Python:

- `"T11, T17"` → `["T11", "T17"]`
- `"-"` → `[]`

In [2]:
df = H.load_tasks(XLSX)
df

,Task,Attività,Tempo,Predecessori
0,T1,Posizionamento telaio sul cavalletto di assemblaggio,40,[]
1,T2,Controllo visivo iniziale del telaio,25,[T1]
2,T3,Installazione serie sterzo,45,[T2]
3,T4,Montaggio forcella,60,[T3]
4,T5,Serraggio forcella con chiave dinamometrica,30,[T4]
5,T6,Installazione attacco manubrio,35,[T5]
6,T7,Montaggio manubrio,45,[T6]
7,T8,Installazione leve freno,40,[T7]
8,T9,Installazione comando cambio,35,[T8]
9,T10,Installazione display LCD,45,[T9]


Dai dati costruiamo i tre dizionari di lavoro: `times`, `predecessors` e `successors`.

In [3]:
times, predecessors, successors = H.build_dicts(df)

print("times (estratto):      ", dict(list(times.items())[:3]))
print("predecessors (estratto):", {k: predecessors[k] for k in ["T1", "T2", "T18"]})
print("successors (estratto):  ", {k: successors[k] for k in ["T1", "T2", "T15"]})

times (estratto):       {'T1': 40, 'T2': 25, 'T3': 45}
predecessors (estratto): {'T1': [], 'T2': ['T1'], 'T18': ['T11', 'T17']}
successors (estratto):   {'T1': ['T2'], 'T2': ['T3', 'T12', 'T14'], 'T15': ['T16', 'T17', 'T21', 'T23']}


**Verifica del Work Content.** La somma dei tempi standard deve valere 1905 s.

In [4]:
WC = sum(times.values())
print(f"Work Content WC = {WC} s  (atteso {H.WC_ATTESO} s)")
assert WC == H.WC_ATTESO, "Il Work Content non coincide con il valore atteso!"


Work Content WC = 1905 s  (atteso 1905 s)


**Verifica di aciclicità.** Prima di applicare gli algoritmi controlliamo, con un ordinamento topologico (algoritmo di Kahn), che il grafo delle precedenze non contenga cicli.

In [5]:
topo = H.topological_order(times, predecessors)
print(f"Nessun ciclo rilevato: {len(topo)} task ordinati topologicamente.")
print("Ordine topologico:", topo)

Nessun ciclo rilevato: 40 task ordinati topologicamente.
Ordine topologico: ['T1', 'T2', 'T3', 'T12', 'T14', 'T4', 'T13', 'T15', 'T5', 'T16', 'T17', 'T21', 'T23', 'T6', 'T28', 'T22', 'T26', 'T30', 'T24', 'T7', 'T29', 'T34', 'T25', 'T32', 'T8', 'T31', 'T27', 'T9', 'T33', 'T10', 'T11', 'T18', 'T19', 'T20', 'T35', 'T36', 'T37', 'T38', 'T39', 'T40']


## 2. Configurazione AS-IS (benchmark)

La configurazione **AS-IS** è il *layout attuale* della linea ed è il nostro **benchmark di partenza**: **6 stazioni** definite da blocchi sequenziali di task, non ottimizzate. È un dato fornito (non calcolato da un algoritmo); le euristiche RPW e LCR verranno poi confrontate con questa baseline. La definizione delle stazioni è in `H.ASIS_STATIONS`.

In [6]:
stazioni_ASIS = H.asis_stations()   # layout reale fornito: 6 stazioni fisse
assign_ASIS = H.station_table(stazioni_ASIS, times, CT)
assign_ASIS

,Stazione,Task assegnati,Tempo stazione (s),Tempo morto (s),Saturazione (%)
0,S1,T1 – T2 – T3 – T4 – T5 – T6 – T7 – T8,320,100,76.19
1,S2,T9 – T10 – T11 – T12 – T13 – T14 – T15,330,90,78.57
2,S3,T16 – T17 – T18 – T19 – T20 – T21 – T22 – T23,380,40,90.48
3,S4,T24 – T25 – T26 – T27 – T28 – T29 – T30 – T31,405,15,96.43
4,S5,T32 – T33 – T34 – T35,210,210,50.0
5,S6,T36 – T37 – T38 – T39 – T40,260,160,61.9
6,Totale,,1905,615,


## 3. Metodo RPW — pesi posizionali

Il **peso posizionale** di un task è

$$PW_i = t_i + \sum_j t_j$$

dove $t_i$ è il tempo del task e $\sum_j t_j$ è la somma dei tempi di **tutti** i suoi successori, diretti e indiretti (trovati con una DFS sul grafo dei successori).

In [7]:
pw = H.positional_weights(times, successors)
rpw_weights = H.rpw_weight_table(times, successors)
rpw_weights

,Task,Tempo,Tempo successori,Peso posizionale,N. successori
Rango,,,,,
1,T1,40,1865,1905,39
2,T2,25,1840,1865,38
3,T14,90,1150,1240,23
4,T15,40,1110,1150,22
5,T3,45,1040,1085,22
6,T4,60,980,1040,21
7,T5,30,950,980,20
8,T21,70,735,805,14
9,T6,35,730,765,15


### Ordinamento RPW

I task vengono ordinati per **peso posizionale decrescente**. A parità di peso si usa come spareggio il tempo (decrescente) e poi l'ID del task, per un risultato deterministico.

In [8]:
ordine_RPW = H.order_rpw(times, pw)
ordine_RPW

['T1',
 'T2',
 'T14',
 'T15',
 'T3',
 'T4',
 'T5',
 'T21',
 'T6',
 'T7',
 'T8',
 'T9',
 'T28',
 'T10',
 'T23',
 'T11',
 'T17',
 'T22',
 'T29',
 'T30',
 'T25',
 'T18',
 'T26',
 'T31',
 'T32',
 'T27',
 'T33',
 'T19',
 'T12',
 'T34',
 'T13',
 'T20',
 'T35',
 'T36',
 'T37',
 'T38',
 'T39',
 'T24',
 'T16',
 'T40']

## 4. Assegnazione RPW alle stazioni

Si apre una stazione e vi si inserisce ripetutamente il task **disponibile** di peso posizionale più alto che non superi il Cycle Time. Un task è disponibile quando i suoi predecessori sono già assegnati o sono nella stazione corrente:

- precedenze: `set(predecessors[task]).issubset(assigned ∪ current_station)`
- tempo: `station_time + times[task] <= CT`

Quando nessun task entra più, la stazione si chiude e se ne apre un'altra.

In [9]:
stazioni_RPW = H.assign_stations(ordine_RPW, times, predecessors, CT)
assign_RPW = H.station_table(stazioni_RPW, times, CT)
assign_RPW

,Stazione,Task assegnati,Tempo stazione (s),Tempo morto (s),Saturazione (%)
0,S1,T1 – T2 – T14 – T15 – T3 – T4 – T5 – T21,400,20,95.24
1,S2,T6 – T7 – T8 – T9 – T28 – T10 – T23 – T11 – T17,420,0,100.0
2,S3,T22 – T29 – T30 – T25 – T18 – T26 – T31 – T32 – T19,420,0,100.0
3,S4,T27 – T33 – T12 – T34 – T13 – T20 – T35 – T36,400,20,95.24
4,S5,T37 – T38 – T39 – T24 – T16 – T40,265,155,63.1
5,Totale,,1905,195,


## 5. Metodo LCR — Largest Candidate Rule

Stessa logica di assegnazione del RPW, ma cambia il criterio di priorità: i task vengono ordinati per **tempo standard decrescente** (spareggio sull'ID del task).

In [10]:
ordine_LCR = H.order_lcr(times)
ordine_LCR

['T14',
 'T21',
 'T27',
 'T33',
 'T36',
 'T4',
 'T18',
 'T25',
 'T28',
 'T11',
 'T17',
 'T35',
 'T37',
 'T38',
 'T23',
 'T26',
 'T3',
 'T7',
 'T10',
 'T22',
 'T31',
 'T32',
 'T39',
 'T1',
 'T8',
 'T15',
 'T24',
 'T29',
 'T30',
 'T34',
 'T6',
 'T9',
 'T13',
 'T16',
 'T19',
 'T40',
 'T5',
 'T12',
 'T20',
 'T2']

### Assegnazione LCR alle stazioni

In [11]:
stazioni_LCR = H.assign_stations(ordine_LCR, times, predecessors, CT)
assign_LCR = H.station_table(stazioni_LCR, times, CT)
assign_LCR

,Stazione,Task assegnati,Tempo stazione (s),Tempo morto (s),Saturazione (%)
0,S1,T1 – T2 – T14 – T3 – T4 – T15 – T21 – T23,420,0,100.0
1,S2,T17 – T26 – T22 – T25 – T27 – T24 – T30 – T32,405,15,96.43
2,S3,T16 – T5 – T28 – T29 – T31 – T34 – T6 – T7 – T8 – T9,405,15,96.43
3,S4,T33 – T10 – T11 – T18 – T19 – T12 – T13 – T20 – T35,415,5,98.81
4,S5,T36 – T37 – T38 – T39 – T40,260,160,61.9
5,Totale,,1905,195,


## 6. Indicatori (KPI) e confronto

Per ogni configurazione calcoliamo, oltre ai valori per-stazione già mostrati — tempo stazione $T_s$, tempo morto $IT_s = CT - T_s$, saturazione $S_s = \frac{T_s}{CT}\cdot 100$ —:

- **Idle Time totale**  $IT = N \cdot CT - WC$
- **Efficienza**  $E = \dfrac{WC}{N \cdot CT}$
- **Balance Delay**  $BD = 1 - E$

con $N$ = numero di stazioni.

In [12]:
configs = {"AS-IS": stazioni_ASIS, "RPW": stazioni_RPW, "LCR": stazioni_LCR}
kpi_tab = H.kpi_comparison(configs, times, CT)
kpi_tab

,Metodo,Numero stazioni,Idle Time (s),Efficienza,Balance Delay
0,AS-IS,6,615,0.7560,0.2440
1,RPW,5,195,0.9071,0.0929
2,LCR,5,195,0.9071,0.0929


> La baseline **AS-IS** impiega **6 stazioni** (efficienza 75.6%, Balance Delay 24.4%). Entrambe le euristiche **RPW** e **LCR** riducono la linea a **5 stazioni** — pari al minimo teorico $\lceil WC/CT \rceil = \lceil 1905/420 \rceil = 5$ — portando l'efficienza al **90.7%** e il Balance Delay al **9.3%**. In questo caso studio RPW e LCR ottengono gli stessi KPI aggregati; ciò che cambia è la *composizione* delle stazioni.

## 7. Esportazione su Excel (multi-foglio)

Salviamo i risultati in un unico file Excel con i seguenti fogli: `Task`, `RPW_weights`, `RPW_order`, `RPW_assignment`, `LCR_order`, `LCR_assignment`, `KPI_comparison` (più `ASIS_assignment` come riferimento della baseline).

In [13]:
def ordine_df(order, etichetta="Task"):
    "DataFrame a due colonne: posizione di priorità e task."
    return pd.DataFrame({"Posizione": range(1, len(order) + 1), etichetta: order})

sheets = {
    "Task": df.assign(
        Predecessori=df["Predecessori"].apply(lambda p: ", ".join(p) if p else "-")
    ),
    "RPW_weights": rpw_weights.reset_index(),
    "RPW_order": ordine_df(ordine_RPW),
    "RPW_assignment": assign_RPW,
    "LCR_order": ordine_df(ordine_LCR),
    "LCR_assignment": assign_LCR,
    "ASIS_assignment": assign_ASIS,
    "KPI_comparison": kpi_tab,
}

out = H.export_excel("risultati_ALB.xlsx", sheets)
print("File Excel creato:", out)

File Excel creato: risultati_ALB.xlsx


---
**Output prodotti:** configurazione benchmark AS-IS, tabella dei pesi posizionali RPW, ordinamento RPW, configurazione RPW, ordinamento LCR, configurazione LCR, tabella KPI per AS-IS/RPW/LCR e il file Excel multi-foglio `risultati_ALB.xlsx`.